# Dependencies

In [ ]:
# !python -m pip install nltk

# Helpful functions

In [1]:
import nltk.internals
from nltk.sem.logic import LogicParser
from nltk.inference import ResolutionProver, Prover9
import regex as re  

In [2]:
parser = LogicParser()
# prover = ResolutionProver()
prover9_path = 'C:\\Program Files (x86)\\Prover9-Mace4\\bin-win32' 
prover = Prover9()
prover.config_prover9(prover9_path)

def translate_fol(fol_text: str):
    # '-' --> '_'
    fol_text = re.sub(r'(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])', '_', fol_text)

    replacements = {
        '∀': 'all ', 
        '∃': 'exists ',
        '∧': '&', 
        '∨': '|',
        '⊕': '^',
        '¬': '-',
        '→': '->', 
        '⟷': '<->',
        '↔': '<->'
    }
    for k, v in replacements.items():
        fol_text = fol_text.replace(k, v)

    # Add dot: "all x (P(x))" --> "all x. (P(x))"
    fol_text = re.sub(r'(all|exists)\s+([a-z0-9]+)\s*', r'\1 \2. ', fol_text)

    # Fix prover9 constants name (eg: "yuri" -> "c_yuri")
    words = re.findall(r'\b[a-z][a-zA-Z0-9_]*\b', fol_text)
    reserved_words = {'all', 'exists', 'u', 'v', 'w', 'x', 'y', 'z'}
    
    for w in set(words):
        if w not in reserved_words:
            fol_text = re.sub(fr'\b{w}\b', f'c_{w}', fol_text)
    return fol_text

def is_valid_fol(fol_list):
    try:
        for line in fol_list:
            if line.strip():
                parser.parse(line)
        return True
    except Exception as e:
        print("Invalid syntax:", e)
        return False

In [3]:
def check_data(data: dict, print_result=True):
    '''
    This function checks if a data sample is correct by checking:
        - If the syntax is correct.
        - If the label prediction (T/F/Uncertain) matches the true label

    Args:
        data: dict ({"id", "natural", "fol", "label"})
    Returns: 
        None/True/False (invalid syntax/correct/wrong prediction)
    '''
    # Read fol strings
    fol_list = data["fol"]
    translated_premises = [translate_fol(p) for p in fol_list[:-1]]
    translated_conclusion = translate_fol(fol_list[-1])

    if (not is_valid_fol(translated_premises) or not is_valid_fol([translated_conclusion])):
        return None
    
    try:
        parsed_premises = [parser.parse(p) for p in translated_premises]
        parsed_conclusion = parser.parse(translated_conclusion)
    except Exception as e:
        print("Error:", e)
        return None

    # Check conclusion
    predicted = "Uncertain"
    try:
        is_true = prover.prove(parsed_conclusion, parsed_premises)
        if is_true:
            predicted = "True"

        is_false = prover.prove(parsed_conclusion.negate(), parsed_premises)
        if is_false:
            predicted = "False"

        if (print_result):
            print("Predicted:", predicted)

        if (predicted == data["label"]):
            return True
    
        return False
    except Exception as e:
        print(data["id"], ": ", e)
        return None


def check_data_list(data_list: list):
    '''
    This function checks if a list of data samples is correct by checking:
        - If the syntax is correct.
        - If the label prediction (T/F/Uncertain) matches the true label

    Args:
        data: list ([{"id", "natural", "fol", "label"}, ...])
    Returns: 
        [...], [...]: list[int], list[int] (list of story_ids that have invalid syntax and are predicted incorrectly, respectively)
    '''
    predicted_wrong_data = [d["id"] for d in data_list if check_data(d, False) == False]
    invalid_data = [d["id"] for d in data_list if check_data(d, False) is None]
    return predicted_wrong_data, invalid_data

# Check your data

### 1. Load your file

Before testing, remember to re-run the block below everytime you make changes in your json file.

In [7]:
import json 

file_name = "Hao.json" # <--- Change here

with open(file_name, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print("Data size:", len(raw_data))
data = {}
for d in raw_data:
    data[d["id"]] = d

Data size: 118


### 2. Check 1 sample

In [ ]:
with open(file_name, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

data = {}
for d in raw_data:
    data[d["id"]] = d

id = "folio_train_429" # <---- Change here

# for f in data[id]["fol"]:
#     print(translate_fol(f))

result = check_data(data[id], True)
print("Label:", data[id]["label"])
if result == True: 
    print(f"✅ Data is Correct!")
elif result == False:
    print(f"❌ Data is predicted Wrong!!!")
elif result is None:
    print("❌ Invalid syntax!!!")

# ⟷

### 3. Check the whole data

In [ ]:
wrong_data, invalid_data = check_data_list(raw_data)
print("Ids of Predicted wrong data:", wrong_data)
print("Ids of invalid syntax data:", invalid_data)

need_fixed_data = wrong_data.copy()
need_fixed_data.extend(invalid_data)
need_fixed_data.sort()

print(need_fixed_data)
print(len(need_fixed_data), " / ", len(raw_data))
# Warning: It may take very long time or get stuck somewhere when running

# Utilities

In [ ]:
base_file = "new.json"
fixed_file = "fixed.json"
src_file = "new.json"

# Read file
with open(base_file, 'r', encoding='utf-8') as f:
    base_data = json.load(f)
base_data_dict = {}
for d in base_data:
    base_data_dict[d["id"]] = d

with open(fixed_file, 'r', encoding='utf-8') as f:
    fixed_data = json.load(f)

# Fix
for d in fixed_data:
    base_data_dict[d["id"]] = d
new_data = list(base_data_dict.values())
    
# Write file
with open(src_file, "w", encoding="utf-8") as f:
        json.dump(new_data, f, ensure_ascii=False, indent=4)


#### Replace fixed data -> original data

In [ ]:
wrong_data = ['folio_train_42', 'folio_train_94', 'folio_train_135', 'folio_train_136', 'folio_train_149', 'folio_train_154', 'folio_train_206', 
           'folio_train_210', 'folio_train_211', 'folio_train_212', 'folio_train_218', 'folio_train_232', 'folio_train_279', 'folio_train_286', 
           'folio_train_308', 'folio_train_323', 'folio_train_325', 'folio_train_338', 'folio_train_339', 'folio_train_393', 'folio_train_402', 
           'folio_train_438', 'folio_train_477', 'folio_train_481', 'folio_train_482', 'folio_train_37', 'folio_train_41', 'folio_train_43', 
           'folio_train_54', 'folio_train_77', 'folio_train_82', 'folio_train_84', 'folio_train_111', 'folio_train_207', 'folio_train_213', 
           'folio_train_259', 'folio_train_260', 'folio_train_332', 'folio_train_381', 'folio_train_421', 'folio_train_422', 'folio_train_423', 
           'folio_train_435', 'folio_train_437', 'folio_train_440', 'folio_train_442', 'folio_train_459', 'folio_train_460', 'folio_train_471', 
           'folio_train_472', 'folio_train_478', 'folio_train_479', 'folio_train_490', 'folio_train_491', 'folio_train_492', 'folio_train_493', 
           'folio_train_494', 'folio_train_499', 'folio_train_500', 'folio_train_520', 'folio_train_521', 'folio_train_522', 'folio_train_532', 
           'folio_train_536', 'folio_train_537', 'folio_train_538', 'folio_train_539', 'folio_train_555', 'folio_train_556', 'folio_train_557', 
           'folio_train_566', 'folio_train_604', 'folio_train_635', 'folio_train_637', 'folio_train_652', 'folio_train_653', 'folio_train_695', 
           'folio_train_702', 'folio_train_726', 'folio_train_727', 'folio_train_733', 'folio_train_738', 'folio_train_739', 'folio_train_740', 
           'folio_train_741', 'folio_train_789', 'folio_train_796', 'folio_train_797', 'folio_train_829', 'folio_train_831', 'folio_train_845', 
           'folio_train_847', 'folio_train_849', 'folio_train_871', 'folio_train_872', 'folio_train_873', 'folio_train_877', 'folio_train_929', 'folio_train_930']

In [6]:
def replace_fixed_data(base_file: str, fixed_file: str, wrong_data=[]):
    with open(base_file, 'r', encoding='utf-8') as f:
        base_data = json.load(f)
    base_data_dict = {}
    for d in base_data:
        base_data_dict[d["id"]] = d

    # Replace fixed data
    with open(fixed_file, 'r', encoding='utf-8') as f:
        fixed_data = json.load(f)
    for d in fixed_data:
        nat_premises = '\n'.join(d["natural"][:-1])
        nat_conclusion = d["natural"][-1]
        fol_premises = '\n'.join(d["fol"][:-1])
        fol_conclusion = d["fol"][-1]
        id = d["id"]

        if id in base_data_dict:
            base_data_dict[id]["nat_premises"] = nat_premises
            base_data_dict[id]["nat_conclusion"] = nat_conclusion
            base_data_dict[id]["fol_premises"] = fol_premises
            base_data_dict[id]["fol_conclusion"] = fol_conclusion
            base_data_dict[id]["label"] = d["label"]

    # Remove unfixed-able data
    for id in wrong_data:
        del base_data_dict[id] 

    new_data = list(base_data_dict.values())

    with open(base_file, "w", encoding="utf-8") as f:
        json.dump(new_data, f, ensure_ascii=False, indent=4)

In [11]:
replace_fixed_data("temp_train.json", "Khoi.json", wrong_data=[])

#### Save + delete wrong train data

In [ ]:
wrong_data_ids = ['folio_train_42', 'folio_train_94', 'folio_train_135', 'folio_train_136', 'folio_train_149', 'folio_train_154', 'folio_train_206', 
           'folio_train_210', 'folio_train_211', 'folio_train_212', 'folio_train_218', 'folio_train_232', 'folio_train_279', 'folio_train_286', 
           'folio_train_308', 'folio_train_323', 'folio_train_325', 'folio_train_338', 'folio_train_339', 'folio_train_393', 'folio_train_402', 
           'folio_train_438', 'folio_train_477', 'folio_train_481', 'folio_train_482', 'folio_train_37', 'folio_train_41', 'folio_train_43', 
           'folio_train_54', 'folio_train_77', 'folio_train_82', 'folio_train_84', 'folio_train_111', 'folio_train_207', 'folio_train_213', 
           'folio_train_259', 'folio_train_260', 'folio_train_332', 'folio_train_381', 'folio_train_421', 'folio_train_422', 'folio_train_423', 
           'folio_train_435', 'folio_train_437', 'folio_train_440', 'folio_train_442', 'folio_train_459', 'folio_train_460', 'folio_train_471', 
           'folio_train_472', 'folio_train_478', 'folio_train_479', 'folio_train_490', 'folio_train_491', 'folio_train_492', 'folio_train_493', 
           'folio_train_494', 'folio_train_499', 'folio_train_500', 'folio_train_520', 'folio_train_521', 'folio_train_522', 'folio_train_532', 
           'folio_train_536', 'folio_train_537', 'folio_train_538', 'folio_train_539', 'folio_train_555', 'folio_train_556', 'folio_train_557', 
           'folio_train_566', 'folio_train_604', 'folio_train_635', 'folio_train_637', 'folio_train_652', 'folio_train_653', 'folio_train_695', 
           'folio_train_702', 'folio_train_726', 'folio_train_727', 'folio_train_733', 'folio_train_738', 'folio_train_739', 'folio_train_740', 
           'folio_train_741', 'folio_train_789', 'folio_train_796', 'folio_train_797', 'folio_train_829', 'folio_train_831', 'folio_train_845', 
           'folio_train_847', 'folio_train_849', 'folio_train_871', 'folio_train_872', 'folio_train_873', 'folio_train_877', 'folio_train_929', 
           'folio_train_930']

In [4]:
import json
def save_wrong_data(base_file: str, dst_file: str, wrong_data_ids=[]):
    with open(base_file, 'r', encoding='utf-8') as f:
        base_data = json.load(f)
    base_data_dict = {}
    for d in base_data:
        base_data_dict[d["id"]] = d

    wrong_data = []
    for id in wrong_data_ids:
        wrong_data.append(base_data_dict[id])
        del base_data_dict[id]
    
    correct_data = list(base_data_dict.values())
    with open(base_file, "w", encoding="utf-8") as f:
        json.dump(correct_data, f, ensure_ascii=False, indent=4)
    with open(dst_file, "w", encoding="utf-8") as f:
        json.dump(wrong_data, f, ensure_ascii=False, indent=4)

In [5]:
save_wrong_data("temp_train.json", "wrong_train_data.json", wrong_data_ids) 